# Lab 04: Tracing & Reproducibility

## Business Context

The CTO asks: *"How do we know the scorer is consistent? Can you prove why it gave Coinbase a 43?"* Enable tracing so every tool call, LLM invocation, and retrieval step is logged. Then tag runs so that rubric version changes are reproducible and auditable.

**After this lab:** Every agent invocation produces a trace — a nested tree of spans showing each LLM call, tool call, and retrieval step with timing. Runs are tagged with rubric version, endpoint, and chunking parameters so you can reproduce any result and compare across rubric versions.

| Attribute | Detail |
|---|---|
| **Key Skills** | MLflow autologging, traces vs spans, experiment tagging, `search_runs`, `search_traces`, version comparison |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$1-2 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-meta-llama-3.1-405b-instruct"

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"LLM      : {LLM_ENDPOINT}")

from databricks.vector_search.client import VectorSearchClient
from langchain_core.tools import tool
from langchain_community.chat_models import ChatDatabricks
from langgraph.prebuilt import create_react_agent

VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

SYSTEM_PROMPT = (
    "You are an IPO Filing Analyzer for a financial research firm. "
    "You have access to S-1 filings from tech IPOs and stock performance data.\n\n"
    "Tools:\n"
    "- search_filings: Search S-1 text for relevant passages. Always include the company name.\n"
    "- get_filing_metadata: Look up filing statistics\n"
    "- get_stock_performance: Look up stock price performance post-IPO\n"
    "- score_clarity: Score a filing section's messaging clarity (1-100)\n"
    "- query_scored_database: Query pre-computed clarity scores joined with stock returns\n\n"
    "Guidelines:\n"
    "- Always use tools to gather data before answering. Never guess.\n"
    "- When searching filings, include the company name AND topic keywords.\n"
    "- Cite the source filing. Structure analysis with clear sections and specific details.\n"
    "- IMPORTANT: You provide financial ANALYSIS, not investment ADVICE."
)

vsc = VectorSearchClient()
_vs_index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

COMPANY_TICKERS = {
    'snowflake': 'SNOW', 'palantir': 'PLTR', 'doordash': 'DASH',
    'coinbase': 'COIN', 'rivian': 'RIVN', 'unity': 'U',
    'roblox': 'RBLX', 'bumble': 'BMBL', 'affirm': 'AFRM',
    'robinhood': 'HOOD', 'toast': 'TOST', 'confluent': 'CFLT',
    'gitlab': 'GTLB', 'hashicorp': 'HCP', 'duolingo': 'DUOL',
    'instacart': 'CART', 'klaviyo': 'KVYO', 'arm': 'ARM',
    'reddit': 'RDDT', 'rubrik': 'RBRK', 'astera': 'ALAB',
    'snow': 'SNOW', 'pltr': 'PLTR', 'dash': 'DASH', 'coin': 'COIN',
    'rivn': 'RIVN', 'rblx': 'RBLX', 'bmbl': 'BMBL', 'afrm': 'AFRM',
    'hood': 'HOOD', 'tost': 'TOST', 'cflt': 'CFLT', 'gtlb': 'GTLB',
    'hcp': 'HCP', 'duol': 'DUOL', 'cart': 'CART', 'kvyo': 'KVYO',
    'rddt': 'RDDT', 'rbrk': 'RBRK', 'alab': 'ALAB',
}

def _extract_tickers(query):
    found = []
    q_lower = query.lower()
    for name, ticker in COMPANY_TICKERS.items():
        if name in q_lower and ticker not in found:
            found.append(ticker)
    return found

def _extract_keywords(query):
    stopwords = {'what', 'did', 'say', 'about', 'in', 'their', 'the', 'and', 'how',
                 'does', 'do', 'compare', 'between', 'from', 'filing', 'filings',
                 's-1', 's1', 'ipo', 'for', 'with', 'that', 'this', 'are', 'was',
                 'were', 'has', 'have', 'had', 'been', 'their', 'they', 'its'}
    company_words = set(COMPANY_TICKERS.keys())
    words = query.lower().replace('?', '').replace(',', '').replace('.', '').split()
    return [w for w in words if w not in stopwords and w not in company_words and len(w) > 2][:5]

@tool
def search_filings(query: str) -> str:
    """Search S-1 filing text for passages relevant to the query.
    Use this for questions about what companies said in their IPO filings.
    Include the company name in your query for best results."""
    tickers = _extract_tickers(query)
    keywords = _extract_keywords(query)
    all_parts = []
    seen_texts = set()

    if tickers:
        for ticker in tickers[:2]:
            # Stage 1: SQL keyword search for precise matches
            if keywords:
                like_clauses = ' OR '.join(
                    [f"LOWER(chunk_text) LIKE '%{kw}%'" for kw in keywords]
                )
                sql = f"""
                    SELECT chunk_text, path FROM {CATALOG}.{SCHEMA}.filing_chunks
                    WHERE LOWER(path) LIKE '%{ticker}%'
                      AND ({like_clauses})
                    ORDER BY chunk_index LIMIT 10
                """
                try:
                    for row in spark.sql(sql).collect():
                        text_key = row.chunk_text[:100]
                        if text_key not in seen_texts:
                            seen_texts.add(text_key)
                            source = row.path.split('/')[-1].replace('-S1.html', '')
                            all_parts.append(f'[Source: {source}]\n{row.chunk_text}')
                except Exception:
                    pass

            # Stage 2: Vector search for semantic matches
            results = _vs_index.similarity_search(
                query_text=query, columns=['chunk_text', 'path'],
                num_results=10, filters={'path LIKE': f'%{ticker}%'},
                query_type='HYBRID',
            )
            for doc in results.get('result', {}).get('data_array', []):
                text_key = doc[0][:100]
                if text_key not in seen_texts:
                    seen_texts.add(text_key)
                    source = doc[1].split('/')[-1].replace('-S1.html', '')
                    all_parts.append(f'[Source: {source}]\n{doc[0]}')
    else:
        results = _vs_index.similarity_search(
            query_text=query, columns=['chunk_text', 'path'],
            num_results=15, query_type='HYBRID',
        )
        for doc in results.get('result', {}).get('data_array', []):
            source = doc[1].split('/')[-1].replace('-S1.html', '')
            all_parts.append(f'[Source: {source}]\n{doc[0]}')

    return '\n\n---\n\n'.join(all_parts[:20]) if all_parts else 'No relevant passages found.'


from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

_uc_fn_names = [
    f"{CATALOG}.{SCHEMA}.get_filing_metadata",
    f"{CATALOG}.{SCHEMA}.get_stock_performance",
]
# Add scoring functions if they exist
for fn in ["score_clarity", "query_scored_database"]:
    try:
        spark.sql(f"DESCRIBE FUNCTION {CATALOG}.{SCHEMA}.{fn}")
        _uc_fn_names.append(f"{CATALOG}.{SCHEMA}.{fn}")
    except Exception:
        pass
_uc_client = DatabricksFunctionClient()
_uc_toolkit = UCFunctionToolkit(function_names=_uc_fn_names, client=_uc_client)
tools = [search_filings] + _uc_toolkit.tools

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)
agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print(f"Agent loaded with {len(tools)} tools: {[t.name for t in tools]}")


## A. Enable MLflow Tracing

**One call** instruments the entire LangChain/LangGraph stack — LLM calls, tool calls, retrieval steps, and prompt rendering all produce spans automatically.

Key concepts:
- **Trace** — the complete record of a single agent invocation (root span + all child spans)
- **Span** — a single unit of work within a trace (e.g., one LLM call, one tool call, one retrieval query)
- **Autologging** — `mlflow.langchain.autolog()` patches LangChain internals so every call creates spans without any extra code

The experiment path organises traces and runs in the MLflow UI under a shared folder.

In [ ]:
import mlflow

# One call instruments everything: LLM calls, tool calls, retrieval
mlflow.langchain.autolog(log_traces=True)

username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/ipo-analyzer/lab-04-tracing"
mlflow.set_experiment(experiment_path)

print(f"Tracing enabled.")
print(f"Experiment: {experiment_path}")

In [ ]:
# Query 1: Research — exercises search_filings + LLM synthesis
q1 = "What does Snowflake say about its competitive advantages in its S-1?"
print(f"Q: {q1}")
print("=" * 70)
r1 = agent.invoke({"messages": [{"role": "user", "content": q1}]})
print(r1["messages"][-1].content)

In [ ]:
# Query 2: Stock lookup — exercises get_stock_performance
q2 = "What was Coinbase's 12-month stock return after its IPO?"
print(f"Q: {q2}")
print("=" * 70)
r2 = agent.invoke({"messages": [{"role": "user", "content": q2}]})
print(r2["messages"][-1].content)

In [ ]:
# Query 3: Clarity scoring — exercises score_clarity
q3 = "Score Coinbase's risk factors section for messaging clarity."
print(f"Q: {q3}")
print("=" * 70)
r3 = agent.invoke({"messages": [{"role": "user", "content": q3}]})
print(r3["messages"][-1].content)

In [ ]:
# Inspect the traces that were auto-captured
experiment = mlflow.get_experiment_by_name(experiment_path)
traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=10,
)

print(f"Captured {len(traces)} traces")
print()

for _, trace in traces.iterrows():
    print(f"  Trace ID : {trace.get('request_id', trace.get('trace_id', 'N/A'))}")
    print(f"  Status   : {trace.get('status', trace.get('state', 'N/A'))}")
    print(f"  Duration : {trace.get('execution_time_ms', 'N/A')} ms")
    print()

## B. Tag Runs for Reproducibility

MLflow **tags** and **params** make runs reproducible:
- **Tags** — mutable key-value metadata (rubric version, environment, requester). Can be updated after the run.
- **Params** — immutable key-value configuration logged once at run start (chunk_size, model name). Cannot be changed after logging.
- **Metrics** — numeric values that can be logged at each step (e.g., answer length, score value).

Tagging with `rubric_version` lets us answer the CTO's question: *"Which version of the rubric produced that Coinbase score?"*

In [ ]:
scoring_query = "Score Coinbase's risk factors section for messaging clarity."

with mlflow.start_run(run_name="coinbase-clarity-v1") as run:
    # Tags: mutable metadata — rubric version, environment, endpoint
    mlflow.set_tags({
        "rubric_version": "v1",
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": "1000",
        "chunk_overlap": "200",
        "query_type": "clarity_scoring",
    })

    # Params: immutable configuration — logged once
    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "num_retrieval_results": 5,
    })

    # Run the scoring query
    result_v1 = agent.invoke({"messages": [{"role": "user", "content": scoring_query}]})
    answer_v1 = result_v1["messages"][-1].content

    # Log a metric: answer length as a proxy for response verbosity
    mlflow.log_metric("answer_length_chars", len(answer_v1))

    run_id_v1 = run.info.run_id
    print(f"Run ID (v1): {run_id_v1}")
    print(f"Answer length: {len(answer_v1)} chars")
    print()
    print(answer_v1)

## C. Compare Rubric Versions

The team proposes a new rubric (v2) that uses smaller chunks (256 chars) to give the scorer more granular context. We simulate this as a tag change and run the same query. Then `mlflow.search_runs()` lets us compare both versions side by side.

> **Tags vs Params for version control:** Tags are the right choice here because they can be updated *after* a run if the rubric is retroactively renamed. Params are immutable and better for configuration values that define *how* the code ran.

In [ ]:
# Simulate v2 rubric: smaller chunk_size for finer-grained context
with mlflow.start_run(run_name="coinbase-clarity-v2") as run:
    mlflow.set_tags({
        "rubric_version": "v2",
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": "256",       # Changed from 1000
        "chunk_overlap": "50",     # Changed from 200
        "query_type": "clarity_scoring",
    })

    mlflow.log_params({
        "llm_endpoint": LLM_ENDPOINT,
        "chunk_size": 256,
        "chunk_overlap": 50,
        "num_retrieval_results": 5,
    })

    # Same query, different rubric context
    result_v2 = agent.invoke({"messages": [{"role": "user", "content": scoring_query}]})
    answer_v2 = result_v2["messages"][-1].content

    mlflow.log_metric("answer_length_chars", len(answer_v2))

    run_id_v2 = run.info.run_id
    print(f"Run ID (v2): {run_id_v2}")
    print(f"Answer length: {len(answer_v2)} chars")
    print()
    print(answer_v2)

In [ ]:
import pandas as pd

# Retrieve both versions with a single filter string
experiment = mlflow.get_experiment_by_name(experiment_path)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.rubric_version IN ('v1', 'v2')",
    order_by=["tags.rubric_version ASC"],
)

# Build a side-by-side comparison table
comparison_cols = [
    "tags.rubric_version",
    "tags.chunk_size",
    "tags.chunk_overlap",
    "tags.llm_endpoint",
    "metrics.answer_length_chars",
    "run_id",
]

comparison = runs_df[comparison_cols].copy()
comparison.columns = ["rubric_version", "chunk_size", "chunk_overlap", "llm_endpoint", "answer_length_chars", "run_id"]

print("=== Rubric Version Comparison ===")
display(comparison)

## Before / After

**Before this lab:** When the CTO asks "Why did Coinbase score 43?", the only answer is "we ran the agent and it said so." There is no record of which rubric version was used, what chunks were retrieved, how long each step took, or whether the result would be the same if you ran it again.

**After this lab:** Every invocation produces a trace — a tree of spans showing the exact LLM call, the tool call, and the retrieval step with timing. Every run is tagged with rubric version, chunk size, and endpoint. `search_runs()` makes the comparison auditable.

In [ ]:
# BEFORE: No tracing — black box invocation
print("=== BEFORE: No tracing (black box) ===")
print("agent.invoke({'messages': [...]})")
print("-> Got answer. But: which chunks were retrieved? How long did the LLM call take?")
print("   What rubric version? Can we reproduce this? NO.")
print()

# AFTER: Trace tree for one query (inspect via mlflow.search_traces)
print("=== AFTER: Trace tree for the v1 scoring query ===")
experiment = mlflow.get_experiment_by_name(experiment_path)
traces = mlflow.search_traces(
    experiment_ids=[experiment.experiment_id],
    max_results=5,
)

if not traces.empty:
    # Show the most recent trace's structure
    latest = traces.iloc[0]
    print(f"Trace ID  : {latest.get('request_id', latest.get('trace_id', 'N/A'))}")
    print(f"Status    : {latest.get('status', latest.get('state', 'N/A'))}")
    print(f"Duration  : {latest.get('execution_time_ms', 'N/A')} ms")
    print()
    print("Span hierarchy (LLM call -> tool call -> retrieval):")
    print("  [root] ReAct agent")
    print("    [span] LLM: choose tool          <-- reasoning step")
    print("    [span] Tool: search_filings       <-- retrieval call")
    print("      [span] VectorSearch.similarity_search")
    print("    [span] Tool: score_clarity        <-- scoring call")
    print("    [span] LLM: synthesize answer     <-- final response")
    print()
    print("All steps logged. Duration, inputs, and outputs captured per span.")
else:
    print("No traces found — ensure mlflow.langchain.autolog(log_traces=True) ran before queries.")

In [ ]:
# Return key results via notebook exit (enables API result inspection)
import json as _json
_results = {}
try:
    _results["traces_captured"] = len(traces)
    _results["v1_answer"] = answer_v1[:500]
    _results["v2_answer"] = answer_v2[:500]
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")